# Alucard - Pixel Art Sprite Generator Training

Train the Alucard diffusion model on Kaggle's free 2x T4 GPUs.

**Dataset:** [evilsocket/alucard-sprites](https://huggingface.co/datasets/evilsocket/alucard-sprites) (312K sprites)

**Strategy:** Kaggle gives 30h/week GPU and 9h/session. We checkpoint every 5 epochs and resume across sessions.

## Setup

In [ ]:
# Install alucard from GitHub
!pip install -q git+https://github.com/evilsocket/alucard.git

In [ ]:
import os
import torch
from pathlib import Path

# Paths
WORK = Path("/kaggle/working")
DATA_DIR = WORK / "data"
CKPT_DIR = WORK / "checkpoints"

DATA_DIR.mkdir(exist_ok=True)
CKPT_DIR.mkdir(exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## Download Dataset

Download from HuggingFace and extract sprites + CLIP embeddings.

In [ ]:
import io
import pyarrow.parquet as pq
from PIL import Image
from huggingface_hub import hf_hub_download

# Check if data already extracted (from previous session)
clip_path = DATA_DIR / "clip_embeddings.pt"
if clip_path.exists():
    existing = len(list(DATA_DIR.glob("sprite_*.png")))
    print(f"Data already present: {existing} sprites")
else:
    print("Downloading dataset from HuggingFace...")
    sprite_idx = 0
    
    for chunk_id in range(6):
        fname = f"data/train-{chunk_id:05d}-of-00006.parquet"
        local = hf_hub_download(
            repo_id="evilsocket/alucard-sprites",
            filename=fname,
            repo_type="dataset",
        )
        table = pq.read_table(local)
        
        for row_idx in range(len(table)):
            img_struct = table.column("image")[row_idx].as_py()
            text = table.column("text")[row_idx].as_py()
            
            img_bytes = img_struct["bytes"]
            img = Image.open(io.BytesIO(img_bytes))
            img.save(DATA_DIR / f"sprite_{sprite_idx:06d}.png")
            (DATA_DIR / f"sprite_{sprite_idx:06d}.txt").write_text(text)
            sprite_idx += 1
        
        print(f"  Chunk {chunk_id + 1}/6: {len(table)} rows ({sprite_idx} total)")
        del table
    
    print(f"\nExtracted {sprite_idx} sprites")
    
    # Pre-compute CLIP embeddings
    print("Computing CLIP embeddings...")
    import open_clip
    
    device = torch.device("cuda")
    clip_model, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
    tokenizer = open_clip.get_tokenizer("ViT-B-32")
    clip_model = clip_model.to(device).eval()
    
    txt_files = sorted(DATA_DIR.glob("sprite_*.txt"))
    all_embs = []
    
    for i in range(0, len(txt_files), 512):
        batch = txt_files[i:i + 512]
        caps = [f.read_text().strip() for f in batch]
        tokens = tokenizer(caps).to(device)
        with torch.no_grad(), torch.amp.autocast("cuda"):
            emb = clip_model.encode_text(tokens)
            emb = (emb / emb.norm(dim=-1, keepdim=True)).cpu().float()
        all_embs.append(emb)
        if (i + 512) % 10000 < 512:
            print(f"  {min(i + 512, len(txt_files))}/{len(txt_files)}")
    
    all_embs = torch.cat(all_embs, dim=0)
    torch.save(all_embs, clip_path)
    print(f"Saved clip_embeddings.pt: {all_embs.shape}")
    
    # Free CLIP model
    del clip_model, tokenizer, all_embs
    torch.cuda.empty_cache()

## Train

Run training with automatic resume from latest checkpoint.

In [ ]:
from alucard.train import train

# Auto-detect resume checkpoint
resume_path = None
for name in ["latest.pt", "best.pt"]:
    p = CKPT_DIR / name
    if p.exists():
        resume_path = str(p)
        print(f"Resuming from: {resume_path}")
        break

if resume_path is None:
    print("Starting fresh training")

train(
    data_dir=str(DATA_DIR),
    output_dir=str(CKPT_DIR),
    epochs=200,
    batch_size=32,       # 2x T4 = 32GB total, can handle bs=32
    lr=1e-4,
    ema_decay=0.9999,
    grad_accum=2,
    save_every=5,        # Checkpoint every 5 epochs for safe resume
    sample_every=10,
    num_workers=2,       # Kaggle has limited CPU
    resume=resume_path,
)

## Check Results

In [ ]:
from IPython.display import display, Image as IPImage
from pathlib import Path

# Show latest samples
sample_dir = CKPT_DIR / "samples"
if sample_dir.exists():
    samples = sorted(sample_dir.glob("*.png"))
    if samples:
        print(f"Latest sample: {samples[-1].name}")
        display(IPImage(filename=str(samples[-1])))

# Show checkpoint info
for name in ["latest.pt", "best.pt"]:
    p = CKPT_DIR / name
    if p.exists():
        ckpt = torch.load(p, map_location="cpu", weights_only=False)
        print(f"{name}: epoch {ckpt['epoch'] + 1}, step {ckpt['global_step']}, "
              f"best_loss {ckpt.get('best_loss', 'N/A')}")
        del ckpt

## Upload Best Model to HuggingFace

Run this cell after training is complete to upload the model.

In [ ]:
# Convert best checkpoint to safetensors and upload
from alucard.convert import convert_to_safetensors
from huggingface_hub import HfApi, login

best_pt = CKPT_DIR / "best.pt"
if best_pt.exists():
    out_safetensors = CKPT_DIR / "alucard_model.safetensors"
    convert_to_safetensors(str(best_pt), str(out_safetensors))
    print(f"Converted to: {out_safetensors}")
    
    # Uncomment and set your token to upload:
    # login(token="hf_YOUR_TOKEN")
    # api = HfApi()
    # api.upload_file(
    #     path_or_fileobj=str(out_safetensors),
    #     path_in_repo="alucard_model.safetensors",
    #     repo_id="evilsocket/alucard",
    # )
    # print("Uploaded to HuggingFace!")
else:
    print("No best.pt found. Train first!")